In [3]:
!rm -rf ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis
!git clone https://github.com/SotirisDimitrakoulakos/ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis.git

Cloning into 'ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 221 (delta 53), reused 213 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 138.07 KiB | 1.34 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [4]:
import pickle
from types import SimpleNamespace

iv3_training = SimpleNamespace()
iv3_validation = SimpleNamespace()
iv3_testing = SimpleNamespace()


with open('/notebooks/data/iv3_training_2.pkl', 'rb') as f:
    iv3_training.X_train, iv3_training.y_train = pickle.load(f)
with open('/notebooks/data/iv3_validation_2.pkl', 'rb') as f:
    iv3_validation.X_val, iv3_validation.y_val = pickle.load(f)
with open('/notebooks/data/iv3_testing_2.pkl', 'rb') as f:
    iv3_testing.X_test, iv3_testing.y_test = pickle.load(f)

In [5]:
# EfficientNet dataset sizes
print("InceptionV3 Training size:", len(iv3_training.X_train), len(iv3_training.y_train))
print("InceptionV3 Validation size:", len(iv3_validation.X_val), len(iv3_validation.y_val))
print("InceptionV3 Testing size:", len(iv3_testing.X_test), len(iv3_testing.y_test))

InceptionV3 Training size: 10633 7
InceptionV3 Validation size: 3545 7
InceptionV3 Testing size: 3544 7


In [2]:
import pandas as pd

balanced_metadata = pd.read_parquet('/notebooks/data/filtered_balanced_dataset_iv3_2.parquet')

In [5]:
# Sanity print: show the first few rows
print(balanced_metadata.head())

# Optional: Print shape and info for additional sanity checks
print("\nShape:", balanced_metadata.shape)
print("\nInfo:")
print(balanced_metadata.info())

      id  gender masterCategory subCategory articleType baseColour  \
0   5271  unisex    accessories        bags   backpacks      green   
1  15653  unisex    accessories        bags   backpacks       blue   
2   7056  unisex    accessories        bags   backpacks      black   
3   4583  unisex    accessories        bags   backpacks       grey   
4  15419  unisex    accessories        bags   backpacks      black   

          season    year   usage  \
0         winter  2015.0  casual   
1           fall  2011.0  casual   
2  summer/spring  2011.0  sports   
3         winter  2015.0  casual   
4           fall  2011.0  casual   

                                  productDisplayName  
0                 Wildcraft Unisex Green Holster Bag  
1  Belkin Unisex Simple Backpack Navy Blue Backpacks  
2                Nike Unisex Trng Max Black Backpack  
3                     Wildcraft Unisex Grey Backpack  
4                 Fila Unisex Silver Black Backpacks  

Shape: (17723, 10)

Info:
<clas

In [4]:
import sys
sys.path.append('/notebooks/ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis/AI_infrastructure/Python_Files')

In [5]:
import importlib
import training
import inceptionv3_model  # import once

importlib.reload(training)
importlib.reload(inceptionv3_model)

2025-06-12 16:12:17.555312: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-12 16:12:17.555365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-12 16:12:17.556392: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-12 16:12:17.562652: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-12 16:12:18.895333: W tensorflow/compiler/tf2

<module 'inceptionv3_model' from '/notebooks/ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis/AI_infrastructure/Python_Files/inceptionv3_model.py'>

In [ ]:
# 4. Train InceptionV3
from training import ClothingClassifierTrainer
from inceptionv3_model import IV3ClothingClassifier


# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# Build and train EfficientNet
iv3net = IV3ClothingClassifier(num_classes_dict)
iv3net_model = iv3net.build_model()

# Set the directory where you want to save/load model and training state
save_dir = '/notebooks/data/IV3Net_2/general'
save_dir_resume = '/notebooks/data/IV3Net_2/general/resume'

iv3net_trainer = ClothingClassifierTrainer(iv3net_model, 'inceptionv3', save_dir=save_dir, save_dir_resume=save_dir_resume)
history_ef = iv3net_trainer.train(iv3_training.X_train, iv3_training.y_train, iv3_validation.X_val,
                                  iv3_validation.y_val, batch_size=16, epochs=50, X_test=iv3_testing.X_test, y_test=iv3_testing.y_test, resume_training=False)

iv3net_trainer.save_model(save_dir)


2025-06-11 22:45:32.310294: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-11 22:45:32.377199: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-11 22:45:32.377430: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

87910968/87910968 [==============================] - 1s 0us/step
Starting fresh training...
Encoded y_train keys and shapes:
masterCategory: (10633,)
subCategory: (10633,)
articleType: (10633,)
baseColour: (10633,)
gender: (10633,)
season: (10633,)
usage: (10633,)
Encoded y_val keys and shapes:
masterCategory: (3545,)
subCategory: (3545,)
articleType: (3545,)
baseColour: (3545,)
gender: (3545,)
season: (3545,)
usage: (3545,)
Batch 0: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Sanity Check Shapes:
X: (16, 299, 299, 3) float32
masterCategory: (16,), int64
subCategory: (16,), int64
articleType: (16,), int64
baseColour: (16,), int64
gender: (16,), int64
season: (16,), int64
usage: (16,), int64
Batch 0: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Epoch 1/50
Batch 317: X shape (16, 299, 299

2025-06-11 22:45:40.113881: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


  1/665 [..............................] - ETA: 1:24:24 - loss: 17.4895 - articleType_loss: 4.4206 - baseColour_loss: 2.8982 - gender_loss: 1.6018 - masterCategory_loss: 1.2492 - season_loss: 1.4470 - subCategory_loss: 3.4129 - usage_loss: 2.4598 - articleType_accuracy: 0.0625 - baseColour_accuracy: 0.0000e+00 - gender_accuracy: 0.2500 - masterCategory_accuracy: 0.3750 - season_accuracy: 0.3750 - subCategory_accuracy: 0.0000e+00 - usage_accuracy: 0.0000e+00

2025-06-11 22:45:43.412393: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f9608008970 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-06-11 22:45:43.412433: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A4000, Compute Capability 8.6
I0000 00:00:1749681943.484931     232 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  3/665 [..............................] - ETA: 22s - loss: 17.3145 - articleType_loss: 4.5896 - baseColour_loss: 2.9269 - gender_loss: 1.3966 - masterCategory_loss: 1.2550 - season_loss: 1.6336 - subCategory_loss: 3.3227 - usage_loss: 2.1902 - articleType_accuracy: 0.0208 - baseColour_accuracy: 0.0625 - gender_accuracy: 0.4792 - masterCategory_accuracy: 0.3958 - season_accuracy: 0.2708 - subCategory_accuracy: 0.0000e+00 - usage_accuracy: 0.1042            Batch 492: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Batch 605: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
  4/665 [..............................] - ETA: 1:41 - loss: 17.2200 - articleType_loss: 4.5484 - baseColour_loss: 2.9314 - gender_loss: 1.4114 - masterCategory_loss: 1.2967 - season_loss: 1.6038 - subCategory_loss: 3.3366 - 

In [10]:
# Resume Training
from training import ClothingClassifierTrainer
from inceptionv3_model import IV3ClothingClassifier


# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# Build and train EfficientNet
iv3net = IV3ClothingClassifier(num_classes_dict)
iv3net_model = iv3net.build_model()

# Set the directory where you want to save/load model and training state
save_dir = '/notebooks/data/IV3Net_2/general'
save_dir_resume = '/notebooks/data/IV3Net_2/general/resume'

iv3net_trainer = ClothingClassifierTrainer(iv3net_model, 'inceptionv3', save_dir=save_dir, save_dir_resume=save_dir_resume)
history_ef = iv3net_trainer.train(iv3_training.X_train, iv3_training.y_train, iv3_validation.X_val,
                                  iv3_validation.y_val, batch_size=16, epochs=50, X_test=iv3_testing.X_test, y_test=iv3_testing.y_test, resume_training=True)

iv3net_trainer.save_model(save_dir)

Model outputs: ['articleType', 'baseColour', 'gender', 'masterCategory', 'season', 'subCategory', 'usage']
Loss keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage']
Resuming training...
Loaded weights from /notebooks/data/EffNet_2/general/resume/efficientnet_final_weights.weights.h5
Resuming from epoch 50
Encoded y_train keys and shapes:
masterCategory: (10633,)
subCategory: (10633,)
articleType: (10633,)
baseColour: (10633,)
gender: (10633,)
season: (10633,)
usage: (10633,)
Encoded y_val keys and shapes:
masterCategory: (3545,)
subCategory: (3545,)
articleType: (3545,)
baseColour: (3545,)
gender: (3545,)
season: (3545,)
usage: (3545,)
Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Sanity Check Shapes:
X: (16, 300, 300, 3) float32
masterCategory: (16,), int64
subCategory: (16,), int64
articleType: (16,), int64
baseColour: (16,), int64


ValueError: No loss keys found in training history. Got keys: []

In [ ]:
# Fine-Tuning

import json
from tensorflow.keras.models import model_from_json
from training import ClothingClassifierTrainer
from inceptionv3_model import IV3ClothingClassifier

# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# 1. Load model architecture from .json file
model_name = 'inceptionv3'
save_dir = '/notebooks/data/IV3Net_2/general'  # path to previously saved model
save_dir_resume = '/notebooks/data/IV3Net_2/fine_tuned/resume'

iv3net = IV3ClothingClassifier(num_classes_dict)
model_arch = iv3net.build_model()

# 2. Load model weights and label encoders
model, label_encoders = ClothingClassifierTrainer.load_model(
    model_name=model_name,
    model_arch=model_arch,
    save_dir=save_dir,
    best_weights=True
)

# 3. Fine-tune the model (unfreeze layers and compile)
model = iv3net.fine_tune(model)

# 4. Create a new trainer for fine-tuning
trainer = ClothingClassifierTrainer(model, model_name, save_dir='/notebooks/data/IV3Net_2/fine_tuned', save_dir_resume=save_dir_resume)
trainer.label_encoders = label_encoders

# 5. Train the fine-tuned model
history_ef_ft = trainer.train(
    iv3_training.X_train,
    iv3_training.y_train,
    iv3_validation.X_val,
    iv3_validation.y_val,
    batch_size=16,
    epochs=20,
    resume_training=False
)

trainer.save_model('/notebooks/data/IV3Net_2/fine_tuned')

2025-06-12 14:10:18.668457: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-12 14:10:18.721470: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-12 14:10:18.721683: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

87910968/87910968 [==============================] - 1s 0us/step


✅ Loaded best weights from /notebooks/data/IV3Net_2/general/inceptionv3_best_weights.weights.h5
Starting fresh training...
Encoded y_train keys and shapes:
masterCategory: (10633,)
subCategory: (10633,)
articleType: (10633,)
baseColour: (10633,)
gender: (10633,)
season: (10633,)
usage: (10633,)
Encoded y_val keys and shapes:
masterCategory: (3545,)
subCategory: (3545,)
articleType: (3545,)
baseColour: (3545,)
gender: (3545,)
season: (3545,)
usage: (3545,)
Batch 0: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Sanity Check Shapes:
X: (16, 299, 299, 3) float32
masterCategory: (16,), int64
subCategory: (16,), int64
articleType: (16,), int64
baseColour: (16,), int64
gender: (16,), int64
season: (16,), int64
usage: (16,), int64
Batch 0: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Epoch 1/20
B

2025-06-12 14:10:28.033389: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


  3/665 [..............................] - ETA: 20s - loss: 2.6000 - articleType_loss: 0.3400 - baseColour_loss: 0.8840 - gender_loss: 0.2393 - masterCategory_loss: 0.0119 - season_loss: 0.6953 - subCategory_loss: 0.1983 - usage_loss: 0.2311 - articleType_accuracy: 0.9167 - baseColour_accuracy: 0.7708 - gender_accuracy: 0.9375 - masterCategory_accuracy: 1.0000 - season_accuracy: 0.7292 - subCategory_accuracy: 0.9167 - usage_accuracy: 0.9167    Batch 596: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Batch 627: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
  4/665 [..............................] - ETA: 40s - loss: 2.6270 - articleType_loss: 0.3430 - baseColour_loss: 0.9055 - gender_loss: 0.3222 - masterCategory_loss: 0.0118 - season_loss: 0.6274 - subCategory_loss: 0.1662 - usage_loss: 0.2

In [9]:
# Resume Fine-Tuning

import json
from tensorflow.keras.models import model_from_json
from training import ClothingClassifierTrainer
from inceptionv3_model import IV3ClothingClassifier

# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# 1. Load model architecture from .json file
model_name = 'inceptionv3'
save_dir = '/notebooks/data/IV3Net_2/general'  # path to previously saved model
save_dir_resume = '/notebooks/data/IV3Net_2/fine_tuned/resume'

iv3net = IV3ClothingClassifier(num_classes_dict)
model_arch = iv3net.build_model()

# 2. Load model weights and label encoders
model, label_encoders = ClothingClassifierTrainer.load_model(
    model_name=model_name,
    model_arch=model_arch,
    save_dir=save_dir,
    best_weights=True
)

# 3. Fine-tune the model (unfreeze layers and compile)
model = iv3net.fine_tune(model)

# 4. Create a new trainer for fine-tuning
trainer = ClothingClassifierTrainer(model, model_name, save_dir='/notebooks/data/IV3Net_2/fine_tuned', save_dir_resume=save_dir_resume)
trainer.label_encoders = label_encoders

# 5. Train the fine-tuned model
history_ef_ft = trainer.train(
    iv3_training.X_train,
    iv3_training.y_train,
    iv3_validation.X_val,
    iv3_validation.y_val,
    batch_size=16,
    epochs=20,
    resume_training=True
)

trainer.save_model('/notebooks/data/IV3Net_2/fine_tuned')

2025-06-12 15:39:38.276275: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-12 15:39:38.310987: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-12 15:39:38.311169: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

✅ Loaded best weights from /notebooks/data/IV3Net_2/general/inceptionv3_best_weights.weights.h5
Resuming training...
Loaded weights from /notebooks/data/IV3Net_2/fine_tuned/resume/inceptionv3_final_weights.weights.h5
Resuming from epoch 17
Encoded y_train keys and shapes:
masterCategory: (10633,)
subCategory: (10633,)
articleType: (10633,)
baseColour: (10633,)
gender: (10633,)
season: (10633,)
usage: (10633,)
Encoded y_val keys and shapes:
masterCategory: (3545,)
subCategory: (3545,)
articleType: (3545,)
baseColour: (3545,)
gender: (3545,)
season: (3545,)
usage: (3545,)
Batch 0: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Sanity Check Shapes:
X: (16, 299, 299, 3) float32
masterCategory: (16,), int64
subCategory: (16,), int64
articleType: (16,), int64
baseColour: (16,), int64
gender: (16,), int64
season: (16,), int64
usage: (16,), int64
Batch 0: X shape (16, 299, 299, 3), y keys: ['mast

2025-06-12 15:39:46.721285: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


  1/665 [..............................] - ETA: 1:10:15 - loss: 2.4763 - articleType_loss: 0.1756 - baseColour_loss: 1.3221 - gender_loss: 0.2473 - masterCategory_loss: 0.0758 - season_loss: 0.4370 - subCategory_loss: 0.0704 - usage_loss: 0.1482 - articleType_accuracy: 1.0000 - baseColour_accuracy: 0.5625 - gender_accuracy: 0.9375 - masterCategory_accuracy: 0.9375 - season_accuracy: 0.8125 - subCategory_accuracy: 0.9375 - usage_accuracy: 1.0000Batch 341: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Batch 91: X shape (16, 299, 299, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
  3/665 [..............................] - ETA: 20s - loss: 2.9795 - articleType_loss: 0.2989 - baseColour_loss: 1.4581 - gender_loss: 0.2247 - masterCategory_loss: 0.0269 - season_loss: 0.6680 - subCategory_loss: 0.1569 - usage_loss: 0.14

TypeError: Object of type float32 is not JSON serializable

In [6]:
from inceptionv3_model import IV3ClothingClassifier

# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

iv3net = IV3ClothingClassifier(num_classes_dict)
model_arch = iv3net.build_model()
model_json = model_arch.to_json()

with open("inceptionv3_architecture.json", "w") as f:
    f.write(model_json)

print("✅ Architecture saved. You do not need to repeat this for fine-tuned version.")


2025-06-12 16:12:32.411921: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-12 16:12:32.458600: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-12 16:12:32.458778: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

✅ Architecture saved. You do not need to repeat this for fine-tuned version.
